# Project 02: Monte Carlo Simulation Engine

**Walkthrough notebook** demonstrating the core capabilities:

1. Simulate single-asset GBM paths and visualise them
2. Simulate a correlated multi-asset portfolio and verify correlation recovery
3. Price a European call option and compare to Black-Scholes
4. Demonstrate variance reduction: antithetic variates vs naive MC
5. Compute portfolio VaR and ES via Monte Carlo

---

In [ ]:
import sys

sys.path.insert(0, "../../../src")

import matplotlib.pyplot as plt
import numpy as np

from risk_analyst.simulation.gbm import simulate_gbm, simulate_gbm_correlated
from risk_analyst.simulation.option_pricing import (
    bs_price,
    price_european_option,
)
from risk_analyst.simulation.risk import mc_portfolio_es, mc_portfolio_var

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100
np.set_printoptions(precision=4)

## 1. Single-Asset GBM Simulation

We simulate $S_t$ under Geometric Brownian Motion:
$$S_{t+\Delta t} = S_t \exp\!\left[\left(\mu - \tfrac{\sigma^2}{2}\right)\Delta t + \sigma\sqrt{\Delta t}\, Z\right], \quad Z \sim N(0,1)$$

The analytic expectation is $\mathbb{E}[S_T] = S_0 \exp(\mu T)$.

In [ ]:
# Parameters
s0 = 100.0
mu = 0.08
sigma = 0.2
T = 1.0
n_steps = 252
n_paths = 10_000
seed = 42

paths = simulate_gbm(s0, mu, sigma, T, n_steps, n_paths, seed)
print(f"Paths shape: {paths.shape}")
print(f"Simulated E[S_T] = {np.mean(paths[:, -1]):.4f}")
print(f"Analytic  E[S_T] = {s0 * np.exp(mu * T):.4f}")

In [ ]:
# Plot a sample of paths
t_grid = np.linspace(0, T, n_steps + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: sample paths
ax = axes[0]
for i in range(50):
    ax.plot(t_grid, paths[i], alpha=0.3, linewidth=0.5)
ax.axhline(s0 * np.exp(mu * T), color="red", linestyle="--", label=r"$E[S_T]$")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Price")
ax.set_title("50 Sample GBM Paths")
ax.legend()

# Right: terminal distribution
ax = axes[1]
ax.hist(paths[:, -1], bins=80, density=True, alpha=0.7, edgecolor="black", linewidth=0.3)
ax.axvline(s0 * np.exp(mu * T), color="red", linestyle="--", label=r"$E[S_T]$")
ax.axvline(np.mean(paths[:, -1]), color="blue", linestyle="--", label="MC mean")
ax.set_xlabel("Terminal Price $S_T$")
ax.set_ylabel("Density")
ax.set_title("Terminal Price Distribution")
ax.legend()

plt.tight_layout()
plt.savefig("../results/gbm_paths.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Correlated Multi-Asset GBM (Cholesky)

For $d$ assets with correlation matrix $\Sigma$, we factorise $\Sigma = L L^\top$ (Cholesky)
and generate correlated Brownian increments: $W = L \cdot Z$ where $Z \sim N(0, I_d)$.

We check that the simulated return correlations match the input.

In [ ]:
n_assets = 3
s0_vec = np.array([100.0, 100.0, 100.0])
mu_vec = np.array([0.05, 0.08, 0.06])
sigma_vec = np.array([0.2, 0.3, 0.25])

target_corr = np.array([
    [1.0, 0.6, 0.3],
    [0.6, 1.0, 0.5],
    [0.3, 0.5, 1.0],
])

corr_paths = simulate_gbm_correlated(
    s0_vec, mu_vec, sigma_vec, target_corr,
    T=1.0, n_steps=252, n_paths=50_000, seed=42,
)
print(f"Shape: {corr_paths.shape}")

In [ ]:
# Compute simulated log-return correlations
log_returns = np.log(corr_paths[:, 1:, :] / corr_paths[:, :-1, :])
flat = log_returns.reshape(-1, n_assets)
sim_corr = np.corrcoef(flat, rowvar=False)

print("Target correlation matrix:")
print(target_corr)
print("\nSimulated correlation matrix:")
print(np.round(sim_corr, 4))
print("\nMax absolute error:", np.max(np.abs(sim_corr - target_corr)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = ["Asset 1", "Asset 2", "Asset 3"]

for ax, mat, title in [
    (axes[0], target_corr, "Target Correlation"),
    (axes[1], sim_corr, "Simulated Correlation"),
]:
    im = ax.imshow(mat, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(n_assets))
    ax.set_yticks(range(n_assets))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_title(title)
    for i in range(n_assets):
        for j in range(n_assets):
            ax.text(j, i, f"{mat[i, j]:.3f}", ha="center", va="center", fontsize=11)

plt.colorbar(im, ax=axes, shrink=0.8)
plt.tight_layout()
plt.savefig("../results/correlation_recovery.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. European Call Pricing: MC vs Black-Scholes

Under the risk-neutral measure, the European call price is:
$$C = e^{-rT}\,\mathbb{E}^Q\!\left[\max(S_T - K, 0)\right]$$

We compare the MC estimate to the Black-Scholes closed-form:
$$C_{BS} = S_0\,\Phi(d_1) - K e^{-rT}\,\Phi(d_2)$$

In [ ]:
s0, K, r, sigma, T = 100, 100, 0.05, 0.2, 1.0

# MC price
mc_price, mc_se, mc_ci = price_european_option(
    s0, K, r, sigma, T, option_type="call", n_paths=500_000, seed=42,
)

# Black-Scholes
bs = bs_price(s0, K, r, sigma, T, "call")

print(f"Black-Scholes price:  {bs:.4f}")
print(f"Monte Carlo price:    {mc_price:.4f}")
print(f"MC standard error:    {mc_se:.4f}")
print(f"95% CI:               ({mc_ci[0]:.4f}, {mc_ci[1]:.4f})")
print(f"Absolute error:       {abs(mc_price - bs):.4f}")
print(f"Error / SE:           {abs(mc_price - bs) / mc_se:.2f}")

In [ ]:
# Convergence study: MC price vs number of paths
path_counts = [100, 500, 1_000, 5_000, 10_000, 50_000, 100_000, 500_000]
mc_prices = []
mc_ses = []

for n in path_counts:
    p, se, _ = price_european_option(s0, K, r, sigma, T, "call", n_paths=n, seed=42)
    mc_prices.append(p)
    mc_ses.append(se)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price convergence
ax = axes[0]
ax.semilogx(path_counts, mc_prices, "o-", label="MC price")
ax.axhline(bs, color="red", linestyle="--", label=f"BS = {bs:.4f}")
ax.fill_between(
    path_counts,
    [p - 2*s for p, s in zip(mc_prices, mc_ses)],
    [p + 2*s for p, s in zip(mc_prices, mc_ses)],
    alpha=0.2, label="+/- 2 SE",
)
ax.set_xlabel("Number of paths")
ax.set_ylabel("Option price")
ax.set_title("MC Price Convergence to Black-Scholes")
ax.legend()

# SE decay (should be O(1/sqrt(n)))
ax = axes[1]
ax.loglog(path_counts, mc_ses, "o-", label="MC SE")
# Reference line: 1/sqrt(n)
ref = mc_ses[0] * np.sqrt(path_counts[0]) / np.sqrt(path_counts)
ax.loglog(path_counts, ref, "--", color="gray", label=r"$O(1/\sqrt{n})$")
ax.set_xlabel("Number of paths")
ax.set_ylabel("Standard error")
ax.set_title("Standard Error Decay")
ax.legend()

plt.tight_layout()
plt.savefig("../results/convergence.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Variance Reduction: Antithetic Variates

**Antithetic variates** exploit the symmetry of the normal distribution.
For each draw $Z$, we also evaluate the payoff at $-Z$.
The average $\hat{C} = \frac{1}{2}(f(Z) + f(-Z))$ has lower variance because
$\text{Cov}(f(Z), f(-Z)) < 0$ for monotone payoffs.

We compare naive MC vs antithetic MC on a European call.

In [ ]:
import sys

sys.path.insert(0, "../src")

# Direct comparison using raw numpy for clarity
s0, K, r, sigma, T = 100, 100, 0.05, 0.2, 1.0
n_paths = 50_000
seed = 42

# Naive MC
rng = np.random.default_rng(seed)
z = rng.standard_normal(n_paths)
discount = np.exp(-r * T)

s_T = s0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * z)
payoffs_naive = discount * np.maximum(s_T - K, 0)
naive_price = np.mean(payoffs_naive)
naive_se = np.std(payoffs_naive, ddof=1) / np.sqrt(n_paths)

# Antithetic
s_T_pos = s0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * z)
s_T_neg = s0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * (-z))
payoffs_anti = discount * (np.maximum(s_T_pos - K, 0) + np.maximum(s_T_neg - K, 0)) / 2
anti_price = np.mean(payoffs_anti)
anti_se = np.std(payoffs_anti, ddof=1) / np.sqrt(n_paths)

bs_ref = bs_price(s0, K, r, sigma, T, "call")
vr_ratio = (naive_se / anti_se) ** 2

print(f"BS price:             {bs_ref:.4f}")
print(f"Naive MC:             {naive_price:.4f}  (SE = {naive_se:.4f})")
print(f"Antithetic MC:        {anti_price:.4f}  (SE = {anti_se:.4f})")
print(f"Variance reduction:   {vr_ratio:.2f}x")

In [ ]:
# Visualise SE comparison across increasing path counts
path_counts = [1_000, 2_000, 5_000, 10_000, 20_000, 50_000, 100_000]
naive_ses = []
anti_ses = []

for n in path_counts:
    rng_loop = np.random.default_rng(seed)
    z_loop = rng_loop.standard_normal(n)

    s_pos = s0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*z_loop)
    s_neg = s0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*(-z_loop))

    pn = discount * np.maximum(s_pos - K, 0)
    pa = discount * (np.maximum(s_pos - K, 0) + np.maximum(s_neg - K, 0)) / 2

    naive_ses.append(np.std(pn, ddof=1) / np.sqrt(n))
    anti_ses.append(np.std(pa, ddof=1) / np.sqrt(n))

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(path_counts, naive_ses, "o-", label="Naive MC")
ax.loglog(path_counts, anti_ses, "s-", label="Antithetic")
ax.set_xlabel("Number of paths")
ax.set_ylabel("Standard error")
ax.set_title("Naive MC vs Antithetic Variates: Standard Error")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig("../results/antithetic_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Portfolio VaR and ES via Monte Carlo

Given historical returns, we resample (bootstrap) to build a simulated
portfolio return distribution, then compute:

- **VaR($\alpha$)**: the $\alpha$-quantile of the loss distribution
- **ES($\alpha$)**: the expected loss conditional on exceeding VaR

$$\text{ES}_\alpha = \mathbb{E}[-R \mid -R \geq \text{VaR}_\alpha]$$

In [ ]:
# Synthetic returns: 3 assets, 500 trading days
rng_data = np.random.default_rng(42)
n_obs, n_assets_pf = 500, 3

# Correlated returns via Cholesky
corr_pf = np.array([[1.0, 0.5, 0.3],
                     [0.5, 1.0, 0.4],
                     [0.3, 0.4, 1.0]])
L_pf = np.linalg.cholesky(corr_pf)
z_pf = rng_data.standard_normal((n_obs, n_assets_pf))
vols = np.array([0.01, 0.015, 0.012])
means = np.array([0.0003, 0.0005, 0.0002])
returns = means + (z_pf @ L_pf.T) * vols

weights = np.array([0.4, 0.35, 0.25])
n_sims = 100_000

for alpha in [0.90, 0.95, 0.99]:
    var = mc_portfolio_var(returns, weights, alpha=alpha, n_sims=n_sims, seed=42)
    es = mc_portfolio_es(returns, weights, alpha=alpha, n_sims=n_sims, seed=42)
    print(f"alpha={alpha:.2f}:  VaR = {var:.6f}  |  ES = {es:.6f}")

In [ ]:
# Visualise the loss distribution with VaR/ES markers
rng_sim = np.random.default_rng(42)
idx = rng_sim.integers(0, n_obs, size=n_sims)
portfolio_returns_sim = returns[idx] @ weights
losses = -portfolio_returns_sim

alpha = 0.95
var_95 = mc_portfolio_var(returns, weights, alpha=0.95, n_sims=n_sims, seed=42)
es_95 = mc_portfolio_es(returns, weights, alpha=0.95, n_sims=n_sims, seed=42)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(losses, bins=100, density=True, alpha=0.6, edgecolor="black", linewidth=0.3, label="Loss dist.")
ax.axvline(var_95, color="red", linewidth=2, label=f"VaR(95%) = {var_95:.4f}")
ax.axvline(es_95, color="darkred", linewidth=2, linestyle="--", label=f"ES(95%) = {es_95:.4f}")

# Shade the tail
tail_mask = losses >= var_95
if np.any(tail_mask):
    tail_losses = losses[tail_mask]
    ax.hist(tail_losses, bins=30, density=True, alpha=0.4, color="red", label="Tail losses")

ax.set_xlabel("Loss (negative return)")
ax.set_ylabel("Density")
ax.set_title("Portfolio Loss Distribution with VaR and ES")
ax.legend()
plt.tight_layout()
plt.savefig("../results/var_es_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

This notebook demonstrated the full Monte Carlo simulation engine:

| Feature | Result |
|---|---|
| GBM simulation | Simulated mean converges to $S_0 e^{\mu T}$ |
| Cholesky correlation | Simulated correlations match input within 0.01 |
| European call pricing | MC price converges to Black-Scholes within 2 SE |
| Antithetic variates | Variance reduction ratio of ~4x on ATM call |
| Portfolio VaR/ES | VaR monotonicity holds, ES > VaR |

All code lives in `src/risk_analyst/simulation/` (shared library) and
`projects/02_monte_carlo_engine/src/model.py` (orchestrator).